In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import os
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS

#load key for hugging face
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

#1. load the doc
loader = PyPDFLoader("/Users/sanvijanvi/Documents/SJ_HomeWork/attention.pdf")
pdfDoc = loader.load()

#2. Chunking docs
seperators = ["\n\n","\n"," ",""]
pdfSplitter = RecursiveCharacterTextSplitter(separators=seperators,chunk_size = 1000, chunk_overlap = 100)
pdfChunks = pdfSplitter.split_documents(pdfDoc)

#3. Embeddings
pdfEmbeder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

#4.  vector db
vectorStore = FAISS.from_documents(documents=pdfChunks, embedding=pdfEmbeder)

In [ ]:
# 5. retriever
base_retriever = vectorStore.as_retriever(search_type = "similarity",search_kwargs = {"k":3})
# base_retriever.invoke(input = "what is attention?")

In [ ]:
# 6 . llm
from langchain_groq.chat_models import ChatGroq

llm = ChatGroq(model = "openai/gpt-oss-20b")

In [ ]:
# 7 .prompt
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(messages=[
    ('system','You are helpful assistant.Please give the answer to query in 2 lines ONLY using context {context}'),
    
    ('human',"{input}")
])

In [ ]:
# 8 retrieval chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser

doc_chain = create_stuff_documents_chain(llm = llm, prompt=prompt, output_parser= StrOutputParser())
rag_chain = create_retrieval_chain(retriever=base_retriever,combine_docs_chain=doc_chain)

rag_chain.invoke(input= {"input":"What is attention?"})



In [ ]:
rag_chain.invoke(input= {"input":"What is formula for it?"})

In [ ]:
rag_chain.invoke(input= {"input":"My name is Vishala"})

In [ ]:
rag_chain.invoke(input= {"input":"What is my name?"})

# history aware and conversation rag

In [ ]:
# 7 .prompt
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

history_awarePrompt = """
    For given chat history and latest question which has referal context in given chat history,
    rewrite the latest question as standalone question which can be understood without chat history.

    Do not answer the question.
    ONLY rewrite if it is needed.

"""

history_prompt = ChatPromptTemplate.from_messages(messages=[
    ('system',history_awarePrompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ('human',"{input}")
])

In [ ]:
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever

history_aware_retriever = create_history_aware_retriever(llm = llm, retriever=base_retriever, prompt= history_prompt)

In [ ]:
system_answer = """
For given latest question, please answer in the given context {context} ONLY.
Answer should not be more than 100 words.

If answer is not avaliable in given context,
please say I dont have enough information on given question.

""" 

answer_prompt = ChatPromptTemplate.from_messages(messages=[
    ('system',system_answer),
    MessagesPlaceholder(variable_name="chat_history"),
    ('human',"{input}")
])

In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser

doc_chain = create_stuff_documents_chain(llm = llm, output_parser=StrOutputParser(),prompt=answer_prompt)
rag_chain = create_retrieval_chain(retriever= history_aware_retriever, combine_docs_chain=doc_chain)

In [ ]:
from langchain_classic.memory.chat_memory import InMemoryChatMessageHistory
store = {}

def getSession_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory

final = RunnableWithMessageHistory(runnable= rag_chain, get_session_history= getSession_history, input_messages_key="input", history_messages_key="chat_history",output_messages_key="answer")

In [ ]:
config1 = {"configurable":{"session_id":"user123"}}

In [ ]:
final.invoke(input = {"input":"what is attention?"},config = config1)

In [ ]:
config2 = {"configurable":{"session_id":"user234"}}
final.invoke(input = {"input":"what is attention?"},config = config2)

In [ ]:
final.invoke(input = {"input":"what are its advanatages?"},config = config1)

In [ ]:
final.invoke(input = {"input":"what is its formula?"},config = config2)